# **Exploring San Francisco MUNI GTFS Data**
**Dataset:** SF MUNI General Transit Feed Specification (GTFS) — Current Feed  
**Source:** San Francisco Municipal Transportation Agency (SFMTA)  

**File Description:** This notebook explores the GTFS data files for San Francisco's MUNI transit system. GTFS (General Transit Feed Specification) is a standardized format for public transportation schedules and geographic information. We will examine each data file, understand its structure, and build visualizations — including route maps, fare analysis, schedule patterns, and network graphs — to gain insight into how San Francisco moves its passengers.

## Setup
Importing all required libraries and defining file paths.

In [ ]:
# ── Library Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Libraries Imported")

In [ ]:
# ── File Paths ────────────────────────────────────────────────────────────────
# Update BASE_DIR if running on a different machine.
BASE_DIR = r"C:\Users\Owner\Documents\GitHub\GTFS-Exploration\GTFS_Data\Native Data\San Francisco (muni_gtfs-current)"

import os
paths = {
    "agency"         : os.path.join(BASE_DIR, "agency.txt"),
    "calendar"       : os.path.join(BASE_DIR, "calendar.txt"),
    "calendar_dates" : os.path.join(BASE_DIR, "calendar_dates.txt"),
    "directions"     : os.path.join(BASE_DIR, "directions.txt"),
    "fare_attributes": os.path.join(BASE_DIR, "fare_attributes.txt"),
    "fare_rules"     : os.path.join(BASE_DIR, "fare_rules.txt"),
    "routes"         : os.path.join(BASE_DIR, "routes.txt"),
    "shapes"         : os.path.join(BASE_DIR, "shapes.txt"),
    "stop_times"     : os.path.join(BASE_DIR, "stop_times.txt"),
    "stops"          : os.path.join(BASE_DIR, "stops.txt"),
    "timepoints"     : os.path.join(BASE_DIR, "timepoints.txt"),
    "trips"          : os.path.join(BASE_DIR, "trips.txt"),
}

# Load all files
dfs = {name: pd.read_csv(path) for name, path in paths.items()}

print("Files Loaded:")
for name, df in dfs.items():
    print(f"  {name:<20} → {df.shape[0]:>7,} rows  ×  {df.shape[1]:>3} cols")

---
## 1. Agency
**File:** `agency.txt`  
**Description:** Identifies the transit agency or agencies providing the data. For SF MUNI, this will be a single-row file describing the San Francisco Municipal Transportation Agency (SFMTA). It contains the agency name, URL, timezone, and contact information. This file is primarily metadata — no visualization needed, but we display it for reference.

In [ ]:
agency_df = dfs["agency"]
print("=" * 60)
print("AGENCY FILE")
print("=" * 60)
print(f"Shape        : {agency_df.shape}")
print(f"Columns      : {list(agency_df.columns)}")
print()
display(agency_df)

---
## 2. Routes
**File:** `routes.txt`  
**Description:** Defines each transit route — bus lines, cable cars, metro lines, and streetcars. Each row is a unique route with a short name (e.g., "1", "N", "F"), a longer description, a route type code (bus = 3, tram = 0, cable car = 5, etc.), and optional color-coding used in maps and apps.

**Route Type Codes (GTFS Standard):**
- `0` = Tram / Streetcar
- `1` = Subway / Metro
- `3` = Bus
- `5` = Cable Car

In [ ]:
routes_df = dfs["routes"]
print("=" * 60)
print("ROUTES FILE — Overview")
print("=" * 60)
print(f"Total Routes  : {len(routes_df)}")
print(f"Columns       : {list(routes_df.columns)}")
print()
print("Route Type Distribution:")
type_labels = {0: "Tram/Streetcar", 1: "Subway/Metro", 3: "Bus", 5: "Cable Car"}
type_counts = routes_df["route_type"].value_counts().rename(index=type_labels)
print(type_counts.to_string())
print()
print("Sample rows:")
display(routes_df[["route_id", "route_short_name", "route_long_name", "route_type"]].head(10))

In [ ]:
# ── Visualization: Route Type Distribution (Donut Chart) ──────────────────────
type_labels = {0: "Tram/Streetcar", 1: "Subway/Metro", 3: "Bus", 5: "Cable Car"}
type_counts = routes_df["route_type"].map(type_labels).value_counts()

colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]
fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    type_counts,
    labels=type_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=colors[:len(type_counts)],
    wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2),
    textprops={'fontsize': 11}
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_color("white")
    at.set_fontweight("bold")
ax.set_title("SF MUNI — Route Types", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualization: Route Colors (if available) ────────────────────────────────
# GTFS stores route colors as hex strings (without '#'). Let's render them.
if "route_color" in routes_df.columns:
    colored = routes_df[["route_short_name", "route_color", "route_text_color"]].dropna(subset=["route_color"])
    colored = colored[colored["route_color"].str.strip() != ""]
    
    if len(colored) > 0:
        # Show a color swatch table for first 30 routes
        sample = colored.head(30).copy()
        sample["hex"] = "#" + sample["route_color"].str.strip()
        
        fig, ax = plt.subplots(figsize=(12, max(3, len(sample) * 0.35)))
        ax.set_xlim(0, 1)
        ax.set_ylim(0, len(sample))
        ax.axis("off")
        ax.set_title("Route Color Palette (sample)", fontsize=13, fontweight="bold")
        
        for i, (_, row) in enumerate(sample.iterrows()):
            y = len(sample) - i - 0.5
            rect = mpatches.FancyBboxPatch((0.05, y - 0.35), 0.15, 0.7,
                                           boxstyle="round,pad=0.02",
                                           facecolor=row["hex"], edgecolor="#ccc", linewidth=0.5)
            ax.add_patch(rect)
            ax.text(0.25, y, f"Route {row['route_short_name']}  ({row['hex']})",
                    va="center", fontsize=9, color="#222")
        plt.tight_layout()
        plt.show()
    else:
        print("No route color data found.")
else:
    print("route_color column not present.")

---
## 3. Stops
**File:** `stops.txt`  
**Description:** Contains every physical bus stop, metro station, or cable car stop in the MUNI system. Each row has a unique stop ID, a name, and geographic coordinates (latitude/longitude). Some stops also include wheelchair accessibility information. This file is foundational — it anchors the entire transit network in physical space.

In [ ]:
stops_df = dfs["stops"]
print("=" * 60)
print("STOPS FILE — Overview")
print("=" * 60)
print(f"Total Stops   : {len(stops_df):,}")
print(f"Columns       : {list(stops_df.columns)}")
print()
print(f"Lat range     : {stops_df['stop_lat'].min():.4f} → {stops_df['stop_lat'].max():.4f}")
print(f"Lon range     : {stops_df['stop_lon'].min():.4f} → {stops_df['stop_lon'].max():.4f}")
print()
display(stops_df.head(5))

In [ ]:
# ── Visualization: Stop Scatter Map ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 9))
ax.scatter(
    stops_df["stop_lon"],
    stops_df["stop_lat"],
    s=2, alpha=0.4, color="#1565C0", linewidths=0
)
ax.set_title("SF MUNI — All Stop Locations", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude", fontsize=10)
ax.set_ylabel("Latitude", fontsize=10)
ax.set_facecolor("#F5F5F5")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualization: Wheelchair Accessibility ────────────────────────────────────
if "wheelchair_boarding" in stops_df.columns:
    wb_labels = {0: "No Info", 1: "Accessible", 2: "Not Accessible"}
    wb_counts = stops_df["wheelchair_boarding"].map(wb_labels).value_counts()
    
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(wb_counts.index, wb_counts.values,
                   color=["#4CAF50", "#F44336", "#9E9E9E"][:len(wb_counts)],
                   edgecolor="white", linewidth=1.5)
    for bar, val in zip(bars, wb_counts.values):
        ax.text(val + 5, bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=10)
    ax.set_title("Stop Wheelchair Accessibility", fontsize=13, fontweight="bold")
    ax.set_xlabel("Number of Stops")
    ax.set_xlim(0, wb_counts.max() * 1.15)
    ax.grid(axis="x", alpha=0.4)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print("wheelchair_boarding column not found.")

---
## 4. Trips
**File:** `trips.txt`  
**Description:** A trip is a single journey along a route — e.g., one bus completing its full run from start to end. Each row links a trip to a route, a service calendar (weekday/weekend), a headsign (destination sign shown on the vehicle), a shape (geographic path), and a direction (inbound/outbound). This file bridges routes with the actual scheduled runs.

In [ ]:
trips_df = dfs["trips"]
print("=" * 60)
print("TRIPS FILE — Overview")
print("=" * 60)
print(f"Total Trips   : {len(trips_df):,}")
print(f"Unique Routes : {trips_df['route_id'].nunique()}")
print(f"Columns       : {list(trips_df.columns)}")
print()
display(trips_df.head(5))

In [ ]:
# ── Visualization: Trips per Route (Top 25) ────────────────────────────────────
trips_per_route = trips_df.groupby("route_id").size().sort_values(ascending=False).head(25)

# Attempt to label with short route names
if "route_short_name" in routes_df.columns:
    name_map = routes_df.set_index("route_id")["route_short_name"].to_dict()
    trips_per_route.index = [name_map.get(r, r) for r in trips_per_route.index]

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = sns.color_palette("Blues_r", len(trips_per_route))
bars = ax.bar(trips_per_route.index, trips_per_route.values, color=colors_bar, edgecolor="white")
ax.set_title("Top 25 Routes by Number of Scheduled Trips", fontsize=13, fontweight="bold")
ax.set_xlabel("Route", fontsize=10)
ax.set_ylabel("Number of Trips", fontsize=10)
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualization: Trip Direction Split ────────────────────────────────────────
if "direction_id" in trips_df.columns:
    dir_counts = trips_df["direction_id"].map({0: "Outbound (0)", 1: "Inbound (1)"}).value_counts()
    
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(dir_counts.index, dir_counts.values, color=["#1976D2", "#F57C00"], width=0.5, edgecolor="white")
    for i, v in enumerate(dir_counts.values):
        ax.text(i, v + 20, f"{v:,}", ha="center", fontsize=11, fontweight="bold")
    ax.set_title("Trips by Direction", fontsize=13, fontweight="bold")
    ax.set_ylabel("Trip Count")
    ax.grid(axis="y", alpha=0.4)
    sns.despine()
    plt.tight_layout()
    plt.show()

---
## 5. Stop Times
**File:** `stop_times.txt`  
**Description:** The most granular file in GTFS — every scheduled arrival and departure time for every stop on every trip. This is typically the largest file. Each row says: "on trip X, vehicle arrives at stop Y at time Z." This file enables schedule analysis, frequency calculations, and headway computation.

> ⚠️ This file can be very large (100k–1M+ rows). We sample it for heavy computations.

In [ ]:
st_df = dfs["stop_times"]
print("=" * 60)
print("STOP TIMES FILE — Overview")
print("=" * 60)
print(f"Total Records  : {len(st_df):,}")
print(f"Unique Trips   : {st_df['trip_id'].nunique():,}")
print(f"Unique Stops   : {st_df['stop_id'].nunique():,}")
print(f"Columns        : {list(st_df.columns)}")
print()
display(st_df.head(5))

In [ ]:
# ── Visualization: Stops per Trip Distribution ────────────────────────────────
stops_per_trip = st_df.groupby("trip_id")["stop_id"].count()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(stops_per_trip, bins=40, color="#1976D2", edgecolor="white", alpha=0.85)
ax.axvline(stops_per_trip.median(), color="#FF5722", linestyle="--", linewidth=1.8,
           label=f"Median: {stops_per_trip.median():.0f} stops")
ax.axvline(stops_per_trip.mean(), color="#4CAF50", linestyle="--", linewidth=1.8,
           label=f"Mean: {stops_per_trip.mean():.1f} stops")
ax.set_title("Distribution of Stops per Trip", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Stops")
ax.set_ylabel("Frequency (Trips)")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualization: Service Activity by Hour of Day ────────────────────────────
# GTFS times can exceed 24:00 for overnight trips (e.g., 25:30:00)
def parse_gtfs_hour(time_str):
    try:
        return int(str(time_str).split(":")[0])
    except:
        return np.nan

# Sample for speed (use full dataset if memory allows)
sample_st = st_df.sample(min(100_000, len(st_df)), random_state=42)
sample_st["dep_hour"] = sample_st["departure_time"].apply(parse_gtfs_hour)

# Cap at hour 29 (GTFS max reasonable)
hour_counts = sample_st[sample_st["dep_hour"].between(0, 29)]["dep_hour"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hour_counts.index, hour_counts.values, color="#0D47A1", edgecolor="white", width=0.8)
ax.set_title("Scheduled Departures by Hour of Day", fontsize=13, fontweight="bold")
ax.set_xlabel("Hour (0 = midnight, 24+ = next-day overnight service)")
ax.set_ylabel("Departure Count (sampled)")
ax.set_xticks(range(0, 30))
ax.grid(axis="y", alpha=0.35)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualization: Busiest Stops (by scheduled arrivals) ─────────────────────
top_stops = st_df["stop_id"].value_counts().head(20)

# Map stop IDs to names
stop_name_map = stops_df.set_index("stop_id")["stop_name"].to_dict()
top_stops.index = [stop_name_map.get(s, str(s)) for s in top_stops.index]

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = sns.color_palette("Reds_r", len(top_stops))
ax.barh(top_stops.index[::-1], top_stops.values[::-1], color=colors_bar[::-1], edgecolor="white")
ax.set_title("Top 20 Busiest Stops (by Scheduled Stop Events)", fontsize=13, fontweight="bold")
ax.set_xlabel("Scheduled Stop Events")
ax.grid(axis="x", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

---
## 6. Calendar
**File:** `calendar.txt`  
**Description:** Defines service patterns using a weekly schedule. Each row is a `service_id` with boolean flags for each day of the week (Monday–Sunday), plus a start and end date for when that service is active. For example, a "weekday" service runs Mon–Fri, while a "weekend" service runs Sat–Sun. This tells the system which trips are running on which days.

In [ ]:
cal_df = dfs["calendar"]
print("=" * 60)
print("CALENDAR FILE — Overview")
print("=" * 60)
print(f"Service IDs   : {len(cal_df)}")
print(f"Columns       : {list(cal_df.columns)}")
print()
display(cal_df)

In [ ]:
# ── Visualization: Service Day Heatmap ────────────────────────────────────────
days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
available_days = [d for d in days if d in cal_df.columns]

if available_days:
    heatmap_data = cal_df.set_index("service_id")[available_days]
    
    fig, ax = plt.subplots(figsize=(9, max(3, len(heatmap_data) * 0.5 + 1)))
    sns.heatmap(
        heatmap_data,
        annot=True, fmt="d", cmap="Blues",
        cbar=False, linewidths=0.5, linecolor="#eee",
        ax=ax
    )
    ax.set_title("Service Calendar — Active Days per Service ID", fontsize=13, fontweight="bold")
    ax.set_xlabel("Day of Week")
    ax.set_ylabel("Service ID")
    ax.set_xticklabels([d.capitalize() for d in available_days], rotation=30)
    plt.tight_layout()
    plt.show()

---
## 7. Calendar Dates
**File:** `calendar_dates.txt`  
**Description:** Provides exceptions to the regular weekly schedule defined in `calendar.txt`. Each row specifies a service ID, a specific date, and an exception type: `1` = service added on this date, `2` = service removed on this date. This file handles holidays, special events, and schedule disruptions.

In [ ]:
cd_df = dfs["calendar_dates"]
cd_df["date"] = pd.to_datetime(cd_df["date"], format="%Y%m%d", errors="coerce")
cd_df["exception_label"] = cd_df["exception_type"].map({1: "Service Added", 2: "Service Removed"})

print("=" * 60)
print("CALENDAR DATES FILE — Overview")
print("=" * 60)
print(f"Total Exceptions  : {len(cd_df):,}")
print(f"Date Range        : {cd_df['date'].min().date()} → {cd_df['date'].max().date()}")
print()
print("Exception Type Breakdown:")
print(cd_df["exception_label"].value_counts().to_string())
print()
display(cd_df.head(5))

In [ ]:
# ── Visualization: Exceptions Over Time ───────────────────────────────────────
cd_monthly = cd_df.groupby([cd_df["date"].dt.to_period("M"), "exception_label"]).size().unstack(fill_value=0)
cd_monthly.index = cd_monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(12, 4))
cd_monthly.plot(kind="bar", ax=ax, color=["#4CAF50", "#F44336"], edgecolor="white", width=0.7)
ax.set_title("Calendar Exceptions by Month", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Exceptions")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Exception Type")
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

---
## 8. Shapes
**File:** `shapes.txt`  
**Description:** Defines the geographic paths that vehicles travel for each route shape. Each row is a single lat/lon point, with a sequence number, belonging to a `shape_id`. Connecting these points in order draws the route line on a map. Multiple trips can share the same shape.

In [ ]:
shapes_df = dfs["shapes"]
print("=" * 60)
print("SHAPES FILE — Overview")
print("=" * 60)
print(f"Total Points   : {len(shapes_df):,}")
print(f"Unique Shapes  : {shapes_df['shape_id'].nunique()}")
print(f"Columns        : {list(shapes_df.columns)}")
print()
display(shapes_df.head(5))

In [ ]:
# ── Visualization: All Route Shapes Overlay Map ───────────────────────────────
# Draw each shape as a line on the map — reveals the full transit network geometry
fig, ax = plt.subplots(figsize=(9, 10))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

cmap = plt.cm.get_cmap("plasma")
shape_ids = shapes_df["shape_id"].unique()
n = len(shape_ids)

for i, sid in enumerate(shape_ids):
    seg = shapes_df[shapes_df["shape_id"] == sid].sort_values("shape_pt_sequence")
    ax.plot(seg["shape_pt_lon"], seg["shape_pt_lat"],
            linewidth=0.4, alpha=0.4, color=cmap(i / n))

ax.set_title("SF MUNI — All Route Shapes", fontsize=14, fontweight="bold", color="white")
ax.set_xlabel("Longitude", fontsize=9, color="#aaa")
ax.set_ylabel("Latitude", fontsize=9, color="#aaa")
ax.tick_params(colors="#aaa")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")
plt.tight_layout()
plt.show()

---
## 9. Fare Attributes & Fare Rules
**Files:** `fare_attributes.txt` and `fare_rules.txt`  
**Description:**  
- `fare_attributes.txt` defines fare products — price, currency, payment method (on board vs. before boarding), and transfer rules.
- `fare_rules.txt` maps fares to routes (and optionally origin/destination zones), specifying which fare applies to which journey.

Together they describe the ticketing and pricing structure of the MUNI system.

In [ ]:
fare_attr_df = dfs["fare_attributes"]
fare_rules_df = dfs["fare_rules"]

print("=" * 60)
print("FARE ATTRIBUTES")
print("=" * 60)
print(f"Unique Fare IDs : {len(fare_attr_df)}")
display(fare_attr_df)

print()
print("=" * 60)
print("FARE RULES")
print("=" * 60)
print(f"Total Fare Rules  : {len(fare_rules_df):,}")
print(f"Unique Routes     : {fare_rules_df['route_id'].nunique() if 'route_id' in fare_rules_df.columns else 'N/A'}")
display(fare_rules_df.head(10))

In [ ]:
# ── Visualization: Fare Price Overview ────────────────────────────────────────
if "price" in fare_attr_df.columns and len(fare_attr_df) > 0:
    fare_attr_df["price"] = pd.to_numeric(fare_attr_df["price"], errors="coerce")

    fig, ax = plt.subplots(figsize=(7, max(3, len(fare_attr_df) * 0.6 + 1)))
    bars = ax.barh(
        fare_attr_df["fare_id"].astype(str),
        fare_attr_df["price"],
        color="#1565C0", edgecolor="white"
    )
    for bar, val in zip(bars, fare_attr_df["price"]):
        ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                f"${val:.2f}", va="center", fontsize=10)
    currency = fare_attr_df["currency_type"].iloc[0] if "currency_type" in fare_attr_df.columns else "USD"
    ax.set_title(f"Fare Prices ({currency})", fontsize=13, fontweight="bold")
    ax.set_xlabel("Price")
    ax.set_xlim(0, fare_attr_df["price"].max() * 1.25)
    ax.grid(axis="x", alpha=0.4)
    sns.despine()
    plt.tight_layout()
    plt.show()
    
    # Transfer rules
    if "transfers" in fare_attr_df.columns:
        transfer_labels = {0: "No Transfers", 1: "1 Transfer", 2: "2 Transfers", -1: "Unlimited"}
        print("\nTransfer Policies:")
        for _, row in fare_attr_df.iterrows():
            t = transfer_labels.get(int(row["transfers"]), str(row["transfers"]))
            print(f"  {row['fare_id']}: {t}")

---
## 10. Directions
**File:** `directions.txt`  
**Description:** A non-standard GTFS extension (used by some agencies including SFMTA) that provides human-readable direction labels for routes. It maps a `route_id` and `direction_id` (0 or 1) to a descriptive name like "Inbound" or "Outbound", or specific terminus labels like "To Downtown" vs "To the Beach." This supplements the standard `direction_id` field in trips.

In [ ]:
dir_df = dfs["directions"]
print("=" * 60)
print("DIRECTIONS FILE — Overview")
print("=" * 60)
print(f"Total Records  : {len(dir_df):,}")
print(f"Columns        : {list(dir_df.columns)}")
print()
display(dir_df.head(20))

In [ ]:
# ── Analysis: Most Common Direction Labels ────────────────────────────────────
dir_col = [c for c in dir_df.columns if "direction" in c.lower() and "id" not in c.lower()]
if dir_col:
    label_counts = dir_df[dir_col[0]].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(label_counts.index[::-1], label_counts.values[::-1],
            color="#5C6BC0", edgecolor="white")
    ax.set_title("Most Common Direction Labels", fontsize=13, fontweight="bold")
    ax.set_xlabel("Number of Routes")
    ax.grid(axis="x", alpha=0.4)
    sns.despine()
    plt.tight_layout()
    plt.show()

---
## 11. Timepoints
**File:** `timepoints.txt`  
**Description:** Another non-standard GTFS extension specifying which stops in a trip are "timepoints" — designated stops where vehicles are expected to maintain schedule adherence. Stops between timepoints are approximate. This is operationally important: drivers must not be early at timepoints but may be flexible at non-timepoints. Used for performance tracking.

In [ ]:
tp_df = dfs["timepoints"]
print("=" * 60)
print("TIMEPOINTS FILE — Overview")
print("=" * 60)
print(f"Total Records  : {len(tp_df):,}")
print(f"Columns        : {list(tp_df.columns)}")
print()
display(tp_df.head(10))

if "timepoint" in tp_df.columns or any("time" in c.lower() for c in tp_df.columns):
    tp_col = [c for c in tp_df.columns if "timepoint" in c.lower()][0] if any("timepoint" in c.lower() for c in tp_df.columns) else tp_df.columns[-1]
    print(f"\nTimepoint value distribution (column: '{tp_col}'):")
    print(tp_df[tp_col].value_counts().to_string())

---
## 12. Network Graph: Stop Connectivity
**Description:** This section constructs a **social/transit network graph** where:
- **Nodes** = transit stops
- **Edges** = sequential stop pairs that appear together on a trip

This reveals which stops are highly connected hubs versus peripheral endpoints, and which corridors carry the most service. Node size reflects how many routes serve a stop (degree centrality).

> For readability, we sample a subset of routes.

In [ ]:
# ── Build Stop-to-Stop Network ────────────────────────────────────────────────
# Use a manageable subset: pick trips from the top N routes
TOP_N_ROUTES = 15

top_route_ids = trips_df.groupby("route_id").size().sort_values(ascending=False).head(TOP_N_ROUTES).index.tolist()
top_trip_ids = trips_df[trips_df["route_id"].isin(top_route_ids)]["trip_id"].unique()

# Filter stop_times to those trips
sub_st = st_df[st_df["trip_id"].isin(top_trip_ids)].sort_values(["trip_id", "stop_sequence"])

# Build edges: consecutive stop pairs within each trip
edges = []
for trip_id, group in sub_st.groupby("trip_id"):
    stop_list = group["stop_id"].tolist()
    for i in range(len(stop_list) - 1):
        edges.append((stop_list[i], stop_list[i + 1]))

# Edge weights = number of times a connection appears
edge_counts = Counter(edges)

# Build graph
G = nx.Graph()
for (u, v), weight in edge_counts.items():
    if G.has_edge(u, v):
        G[u][v]["weight"] += weight
    else:
        G.add_edge(u, v, weight=weight)

print(f"Graph built from top {TOP_N_ROUTES} routes")
print(f"  Nodes (stops)  : {G.number_of_nodes():,}")
print(f"  Edges (links)  : {G.number_of_edges():,}")

In [ ]:
# ── Visualization: Network Graph (geographic layout) ─────────────────────────
# Position nodes using actual lat/lon coordinates
pos = {}
for node in G.nodes():
    row = stops_df[stops_df["stop_id"] == node]
    if not row.empty:
        pos[node] = (float(row.iloc[0]["stop_lon"]), float(row.iloc[0]["stop_lat"]))

# Only keep nodes with known positions
nodes_with_pos = [n for n in G.nodes() if n in pos]
H = G.subgraph(nodes_with_pos)

# Compute degree centrality for node sizing
degree = dict(H.degree())
node_sizes = [max(5, degree[n] * 4) for n in H.nodes()]
edge_weights = [H[u][v]["weight"] for u, v in H.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_alpha = [0.1 + 0.5 * (w / max_w) for w in edge_weights]

fig, ax = plt.subplots(figsize=(10, 11))
ax.set_facecolor("#0d1117")
fig.patch.set_facecolor("#0d1117")

# Draw edges
nx.draw_networkx_edges(
    H, pos, ax=ax,
    edge_color=[plt.cm.plasma(0.5 + 0.4 * (w / max_w)) for w in edge_weights],
    width=[0.3 + 1.2 * (w / max_w) for w in edge_weights],
    alpha=0.5
)
# Draw nodes
nx.draw_networkx_nodes(
    H, pos, ax=ax,
    node_size=node_sizes,
    node_color=[degree[n] for n in H.nodes()],
    cmap="YlOrRd",
    alpha=0.9
)

ax.set_title(
    f"SF MUNI Stop Network — Top {TOP_N_ROUTES} Routes\n(node size & color = connection degree)",
    fontsize=13, fontweight="bold", color="white"
)
ax.tick_params(colors="#888")
ax.set_xlabel("Longitude", color="#888", fontsize=8)
ax.set_ylabel("Latitude", color="#888", fontsize=8)
for spine in ax.spines.values():
    spine.set_edgecolor("#333")
plt.tight_layout()
plt.show()

In [ ]:
# ── Top Hub Stops by Network Degree ───────────────────────────────────────────
degree_series = pd.Series(dict(G.degree())).sort_values(ascending=False).head(20)
degree_series.index = [stop_name_map.get(s, str(s)) for s in degree_series.index]

fig, ax = plt.subplots(figsize=(10, 6))
colors_d = sns.color_palette("YlOrRd", len(degree_series))
ax.barh(degree_series.index[::-1], degree_series.values[::-1],
        color=colors_d, edgecolor="white")
ax.set_title("Top 20 Stop Hubs by Network Degree", fontsize=13, fontweight="bold")
ax.set_xlabel("Degree (number of unique connected stops)")
ax.grid(axis="x", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

---
## 13. Route-to-Route Sharing Network
**Description:** A second network graph at the **route level**. Two routes are connected if they **share at least one stop**. This reveals route clusters, transfer opportunities, and which routes act as connectors across the system. Edge weight = number of shared stops.

In [ ]:
# ── Build Route Sharing Graph ──────────────────────────────────────────────────
# Map each trip to its route
trip_to_route = trips_df.set_index("trip_id")["route_id"].to_dict()

# Map each stop to the set of routes that visit it
sub_st_all = st_df.copy()
sub_st_all["route_id"] = sub_st_all["trip_id"].map(trip_to_route)
sub_st_all = sub_st_all.dropna(subset=["route_id"])

stop_routes = sub_st_all.groupby("stop_id")["route_id"].apply(set).to_dict()

# Build edges between routes sharing stops
from itertools import combinations
route_edges = Counter()
for stop, route_set in stop_routes.items():
    route_list = list(route_set)
    for a, b in combinations(sorted(route_list), 2):
        route_edges[(a, b)] += 1

# Build the route graph
RG = nx.Graph()
for (a, b), shared in route_edges.items():
    RG.add_edge(a, b, weight=shared)

# Relabel with short names if possible
if "route_short_name" in routes_df.columns:
    short_map = routes_df.set_index("route_id")["route_short_name"].to_dict()
    RG = nx.relabel_nodes(RG, lambda x: short_map.get(x, str(x)))

print(f"Route Sharing Graph")
print(f"  Nodes (routes) : {RG.number_of_nodes()}")
print(f"  Edges (pairs)  : {RG.number_of_edges():,}")

In [ ]:
# ── Visualization: Route-to-Route Sharing Network ─────────────────────────────
# Use spring layout and show only edges with meaningful sharing
MIN_SHARED = 3  # filter weak connections
RG_filtered = nx.Graph([(u, v, d) for u, v, d in RG.edges(data=True) if d["weight"] >= MIN_SHARED])

r_degree = dict(RG_filtered.degree())
r_sizes = [max(50, r_degree.get(n, 1) * 20) for n in RG_filtered.nodes()]
r_edge_w = [RG_filtered[u][v]["weight"] for u, v in RG_filtered.edges()]
max_rw = max(r_edge_w) if r_edge_w else 1

pos_r = nx.spring_layout(RG_filtered, seed=42, k=1.5)

fig, ax = plt.subplots(figsize=(14, 11))
ax.set_facecolor("#f8f9fa")

nx.draw_networkx_edges(
    RG_filtered, pos_r, ax=ax,
    width=[0.5 + 3 * (w / max_rw) for w in r_edge_w],
    edge_color=r_edge_w, edge_cmap=plt.cm.Blues,
    alpha=0.7
)
nx.draw_networkx_nodes(
    RG_filtered, pos_r, ax=ax,
    node_size=r_sizes,
    node_color=[r_degree.get(n, 0) for n in RG_filtered.nodes()],
    cmap="Oranges", alpha=0.9
)
nx.draw_networkx_labels(
    RG_filtered, pos_r, ax=ax,
    font_size=6, font_weight="bold", font_color="#222"
)

ax.set_title(
    f"Route Sharing Network (min {MIN_SHARED} shared stops)\nNode size = connections to other routes",
    fontsize=13, fontweight="bold"
)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 14. Summary Statistics
A consolidated overview of the entire GTFS dataset.

In [ ]:
print("=" * 65)
print("  SF MUNI GTFS — SYSTEM SUMMARY")
print("=" * 65)
print(f"  Transit Agency     : {dfs['agency']['agency_name'].iloc[0] if 'agency_name' in dfs['agency'].columns else 'SFMTA'}")
print(f"  Total Routes       : {len(routes_df):,}")
print(f"  Total Trips        : {len(trips_df):,}")
print(f"  Total Stops        : {len(stops_df):,}")
print(f"  Total Stop Events  : {len(st_df):,}")
print(f"  Shape Points       : {len(shapes_df):,}")
print(f"  Unique Shapes      : {shapes_df['shape_id'].nunique():,}")
print(f"  Service IDs        : {len(cal_df)}")
print(f"  Calendar Exceptions: {len(cd_df):,}")
print(f"  Fare Products      : {len(fare_attr_df)}")
print(f"  Fare Rules         : {len(fare_rules_df):,}")
print("=" * 65)

# Route type breakdown
print("\nRoute Breakdown by Type:")
type_labels = {0: "Tram/Streetcar", 1: "Subway/Metro", 3: "Bus", 5: "Cable Car"}
for rtype, count in routes_df["route_type"].value_counts().items():
    label = type_labels.get(rtype, f"Type {rtype}")
    print(f"  {label:<20}: {count}")

# Average stops per trip
avg_stops = st_df.groupby("trip_id")["stop_id"].count().mean()
print(f"\nAvg Stops per Trip   : {avg_stops:.1f}")

# Trips per route
avg_trips = trips_df.groupby("route_id").size().mean()
print(f"Avg Trips per Route  : {avg_trips:.1f}")